In [ ]:
# kaggle API를 사용하기 위해 API 키 json 파일 업로드
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"jhw0714","key":"5ebfcd63c0dd4e608eb3f7b6fbbb5ebd"}'}

In [ ]:
# kaggle.json 파일을 적절한 디렉토리로 옮긴다.
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 데이터셋을 다운로드한다.
!kaggle datasets download -d vladimirvorobevv/chatgpt-paraphrases
!unzip -o /content/chatgpt-paraphrases.zip -d /content/sample_data

chatgpt-paraphrases.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  /content/chatgpt-paraphrases.zip
  inflating: /content/sample_data/chatgpt_paraphrases.csv  


In [ ]:
# 파이스파크 설치
!pip install pyspark
!pip install -U -q PyDrive
!apt install openjdk-8-jdk-headless -qq

# 한국어 텍스트 전처리를 도와주는 NLTK 패키지
!pip install nltk

import os
import re
from google.colab import drive
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
drive.mount('/content/drive')

openjdk-8-jdk-headless is already the newest version (8u382-ga-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 10 not upgraded.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Let's import the libraries we will need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pyspark
from pyspark.sql import *
from pyspark.sql.functions import *
# SparkContext is the entry point to any spark functionality.
# SparkConf provides configurations to run a Spark application.
from pyspark import SparkContext, SparkConf

# create the Spark Session
spark = SparkSession.builder.appName("WordCount").getOrCreate()

# create the Spark Context
sc = spark.sparkContext

In [ ]:
data=pd.read_csv('/content/sample_data/chatgpt_paraphrases.csv')

In [ ]:
# 각 문장을 사람이 작성했는지, AI가 작성했는지 태그
category={}
for i in range(len(data)):
    chatgpt=data.iloc[i]["paraphrases"][1:-1].split(', ')
    for j in chatgpt[:1]:
        category[j[1:-1]]='chatgpt'
    category[data.iloc[i]['text']]="human"

In [ ]:
# 사람의 작성한 문장과 AI가 작성한 문장을 분리
totla_paraphrases=pd.DataFrame(category.items(),columns=["text","category"])
human_paraphrases = totla_paraphrases[totla_paraphrases['category'] == 'human']
gpt_paraphrases = totla_paraphrases[totla_paraphrases['category'] == 'chatgpt']

In [ ]:
import nltk

from nltk.stem import PorterStemmer
from nltk.stem import LancasterStemmer
from nltk.stem import RegexpStemmer
from nltk.stem import WordNetLemmatizer

from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize

from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

lemmatizer = WordNetLemmatizer()

use_stopwords = False
english_stop_list = [".", ",", "!", "?", ")", "(", "'", '"', "]", "[", "{", "}", ":", ";", "<", ">", "~", "''", "``"]

if use_stopwords:
  english_stop_list += list(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# 패턴 찾아내기

GPT:

In [ ]:
from pyspark.ml.fpm import FPGrowth

In [ ]:
gpt_wordbasket=[]
for i in range(len(gpt_paraphrases)):
  listofword = word_tokenize(gpt_paraphrases.iloc[i]['text'].lower())
  for w in listofword:
    if w not in english_stop_list:
      gpt_wordbasket.append((i, lemmatizer.lemmatize(w , pos="v")))

In [ ]:
gpt_worddata = pd.DataFrame(gpt_wordbasket, columns=['paraphrases_id', 'word'])
gpt_worddata_spark = spark.createDataFrame(gpt_worddata)
gpt_wordbag  = gpt_worddata_spark.groupBy("paraphrases_id").agg(collect_set('word').alias('words'))

gpt_fpGrowth = FPGrowth(itemsCol="words", minSupport=0.01, minConfidence=0.5)
gpt_model = gpt_fpGrowth.fit(gpt_wordbag)
gpt_wordbag.show(20, truncate=False)

+--------------+-----------------------------------------------------------------------------------------------------------------------+
|paraphrases_id|words                                                                                                                  |
+--------------+-----------------------------------------------------------------------------------------------------------------------+
|0             |[the, indian, market, in, can, detail, procedure, provide, a, for, you, stock, invest]                                 |
|7             |[skilled, geologist, step, i, become, to, a, take, what, can]                                                          |
|19            |[the, google, answer, quora, that, of, users, ask, through, be, abundance, question, behind, easily, reason, what, can]|
|22            |[the, be, 's, of, someone, root, jealousy, what]                                                                       |
|26            |[the, advice, navigate, f

In [ ]:
print("{} frequent itemsets were generated.\n=================================================================".format(gpt_model.freqItemsets.count()))
freqSets = gpt_model.freqItemsets.sort("freq", ascending=False)
freqSets.show(20, truncate=False)

print("{} association rules were generated.\n=================================================================".format(gpt_model.associationRules.count()))
AssoRules = gpt_model.associationRules.sort("support", ascending=False)
AssoRules.show(20, truncate=False)

782 frequent itemsets were generated.
+---------------+------+
|items          |freq  |
+---------------+------+
|[the]          |230968|
|[be]           |213678|
|[be, the]      |142843|
|[what]         |122679|
|[to]           |121090|
|[of]           |117370|
|[in]           |97847 |
|[of, the]      |97624 |
|[what, be]     |97543 |
|[a]            |96524 |
|[what, the]    |85227 |
|[to, be]       |78059 |
|[what, be, the]|76170 |
|[to, the]      |69393 |
|[of, be]       |68588 |
|[in, the]      |66152 |
|[for]          |65507 |
|[and]          |65187 |
|[of, be, the]  |59194 |
|[a, be]        |56402 |
+---------------+------+
only showing top 20 rows

711 association rules were generated.
+-----------+----------+------------------+------------------+-------------------+
|antecedent |consequent|confidence        |lift              |support            |
+-----------+----------+------------------+------------------+-------------------+
|[be]       |[the]     |0.6684965228053427|1.1256

In [ ]:
rules_generated = AssoRules.join(freqSets, AssoRules.consequent == freqSets.items)\
.withColumn("interest",abs(col("confidence") - (col("freq")/gpt_wordbag.count())))\
.select('antecedent', 'consequent', 'confidence', 'support', 'interest').sort("interest", ascending=False, truncate=False)

In [ ]:
freqSets.toPandas()

,items,freq
0,[the],230968
1,[be],213678
2,"[be, the]",142843
3,[what],122679
4,[to],121090
...,...,...
777,"[on, a, to]",3910
778,"[you, what, the]",3909
779,"[use, of]",3894
780,"[take, the]",3892


In [ ]:
rules_generated.toPandas()

,antecedent,consequent,confidence,support,interest
0,"[possible, for, to, be]",[it],0.992447,0.011149,0.922530
1,"[possible, for, to]",[it],0.991106,0.011175,0.921189
2,"[possible, to, be]",[it],0.984228,0.021020,0.914311
3,"[possible, to]",[it],0.978861,0.021074,0.908944
4,"[explain, can]",[you],0.984955,0.010773,0.904041
...,...,...,...,...,...
706,"['s, of]",[be],0.552748,0.016422,0.003333
707,[would],[be],0.546155,0.010755,0.003260
708,[as],[the],0.592071,0.028530,0.001801
709,"[and, of]",[be],0.550555,0.034316,0.001140
